# param recovery

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
os.environ["OMP_NUM_THREADS"] = "1"
import seaborn as sns 
from scipy.io import loadmat
import ast
from scipy.io import loadmat, savemat
import warnings
warnings.filterwarnings("ignore")
import statsmodels.api as sm


In [2]:
outputFolderName = r"param_recovery_4_param_recovery_v5"
outputFolderName_Rhia = r"\\155.100.91.44\d\Code\Rhiannon\BART\RSTD_project"

inputFolderName = r"\\155.100.91.44\d\Data\Nill\BART_param_recovery\old_modeling_trying_for_better_params\param_recovery_3_simulated_fields"

if not os.path.exists(outputFolderName):
    os.makedirs(outputFolderName)

In [ ]:
matFiles = [f for f in os.listdir(inputFolderName) if f.endswith(".mat")]
nPatients = len(matFiles)
rows = []

for pt in range(nPatients):
# for pt in range(5):
    fileName = matFiles[pt]

    ptID = os.path.splitext(fileName)[0]
    ptID = ptID.replace("_TDdataParamRecovery", "")

    print(f"processing pt {pt+1}/{nPatients}: {ptID}")

    matFile = os.path.join(inputFolderName, fileName)
    mat = loadmat(matFile, struct_as_record=False, squeeze_me=True)

    TDdataParamRecovery = mat["TDdataParamRecovery"]

    alphas = np.asarray(TDdataParamRecovery.a, dtype=float)

    nTrials = int(TDdataParamRecovery.nTrials)

    rewardSimulated = np.asarray(TDdataParamRecovery.Reward, dtype=float)
    rewardSimulated = rewardSimulated[:nTrials]

    result_raw = np.asarray(TDdataParamRecovery.resultSimulated, dtype=str)
    result_raw = result_raw[:nTrials]

    # keep only valid trials
    valid = np.isin(result_raw, ["banked", "popped"])
    rewardSimulated = rewardSimulated[valid]
    result_raw = result_raw[valid]

    nTrials_valid = len(result_raw)

    if nTrials_valid < 3:
        print(f"{ptID}: not enough valid trials after filtering")
        continue

    inverseTemperatureRSTD = np.full((len(alphas), len(alphas)), np.nan)
    fit_score_loglik = np.full((len(alphas), len(alphas)), np.nan)

    expectedReward_1d_all = np.full((len(alphas), len(alphas), nTrials_valid), np.nan, dtype=float)
    RewardPE_1d_all = np.full((len(alphas), len(alphas), nTrials_valid), np.nan, dtype=float)
    predictor_all = np.full((len(alphas), len(alphas), nTrials_valid), np.nan, dtype=float)

    # banked = 1, popped = 0
    y = np.array([1 if x == "banked" else 0 for x in result_raw], dtype=float)

    for ap in range(len(alphas) - 1, -1, -1):
        for an in range(len(alphas) - 1, -1, -1):

            RewardPE = np.zeros(nTrials_valid, dtype=float)
            expectedReward = np.zeros(nTrials_valid, dtype=float)

            # predictor used in GLM: value BEFORE current trial outcome
            predictor = np.full(nTrials_valid, np.nan, dtype=float)
            expectedReward_1d = np.full(nTrials_valid, np.nan, dtype=float)
            RewardPE_1d = np.full(nTrials_valid, np.nan, dtype=float)

            predictor[0] = expectedReward[0]
            expectedReward_1d[0] = expectedReward[0]
            RewardPE_1d[0] = RewardPE[0]

            for t in range(1, nTrials_valid):
                # use pre-outcome expected value to predict current outcome
                predictor[t] = expectedReward[t - 1]

                RewardPE[t] = rewardSimulated[t] - expectedReward[t - 1]

                if RewardPE[t] > 0:
                    expectedReward[t] = expectedReward[t - 1] + alphas[ap] * RewardPE[t]
                elif RewardPE[t] < 0:
                    expectedReward[t] = expectedReward[t - 1] + alphas[an] * RewardPE[t]
                else:
                    expectedReward[t] = expectedReward[t - 1]

                expectedReward_1d[t] = expectedReward[t]
                RewardPE_1d[t] = RewardPE[t]

            expectedReward_1d_all[ap, an, :] = expectedReward_1d
            RewardPE_1d_all[ap, an, :] = RewardPE_1d
            predictor_all[ap, an, :] = predictor

            X = sm.add_constant(predictor[1:])
            y_fit = y[1:]

            try:
                model = sm.GLM(
                    y_fit,
                    X,
                    family=sm.families.Binomial(link=sm.families.links.Logit())
                )
                fit_result = model.fit()

                if len(fit_result.params) > 1:
                    inverseTemperatureRSTD[ap, an] = fit_result.params[1]

                fit_score_loglik[ap, an] = fit_result.llf

            except Exception as e:
                print(f"{ptID} | ap={alphas[ap]:.2f}, an={alphas[an]:.2f} failed: {e}")
                continue

    # check if all fits failed
    if np.all(np.isnan(fit_score_loglik)):
        print(f"{ptID}: all fits failed")
        continue

    # choose best alpha pair from log-likelihood
    best_idx = np.unravel_index(np.nanargmax(fit_score_loglik), fit_score_loglik.shape)
    bestAlphaPosIdx, bestAlphaNegIdx = best_idx

    bestAlphaPos = alphas[bestAlphaPosIdx]
    bestAlphaNeg = alphas[bestAlphaNegIdx]

    rows.append({
        "ptID": ptID,
        "fit_alpha_plus": TDdataParamRecovery.bestAlphaPos,
        "fit_alpha_minus": TDdataParamRecovery.bestAlphaNeg,
        "sim_alpha_plus": bestAlphaPos,
        "sim_alpha_minus": bestAlphaNeg
    })

# save to csv
df_alpha_compare = pd.DataFrame(rows)

output_csv = os.path.join(outputFolderName, "alpha_comparison.csv")
df_alpha_compare.to_csv(output_csv, index=False)
print(f"saved: {output_csv}")


processing pt 1/71: 201810
processing pt 2/71: 201811
processing pt 3/71: 201901
processing pt 4/71: 201902r
processing pt 5/71: 201902
processing pt 6/71: 201903
processing pt 7/71: 201905
processing pt 8/71: 201909
processing pt 9/71: 201910
processing pt 10/71: 201911
processing pt 11/71: 201913
processing pt 12/71: 201914
processing pt 13/71: 201915
processing pt 14/71: 202001
processing pt 15/71: 202002
processing pt 16/71: 202003
processing pt 17/71: 202004
processing pt 18/71: 202005
processing pt 19/71: 202006u
processing pt 20/71: 202006
processing pt 21/71: 202007
processing pt 22/71: 202008
processing pt 23/71: 202009
processing pt 24/71: 202011
processing pt 25/71: 202014
processing pt 26/71: 202015
processing pt 27/71: 202016
processing pt 28/71: 202105
processing pt 29/71: 202107
processing pt 30/71: 202110
processing pt 31/71: 202114


# debug

In [ ]:
fields = [f for f in dir(TDdataParamRecovery) if not f.startswith('_')]
print(fields)

['Reward', 'a', 'bestAlphaNeg', 'bestAlphaPos', 'bestExpectedReward', 'bestFitScoreLogLik', 'bestInverseTemperatureRSTD', 'bestPredictor', 'bestRewardPE', 'fit_score_loglik', 'inflate_time', 'inverseTemperatureRSTD', 'is_control', 'nTrials', 'points', 'pointsMinusReward', 'result', 'resultSimulated', 'rewardSimulated', 'trial_type']


In [ ]:
expectedReward.shape

(231,)